# PDF to Audiobook — Chatterbox TTS Voice Cloning
Converts a PDF into chapter-by-chapter MP3s using a cloned voice.

**Before running:**
1. Make sure GPU is enabled: `Runtime → Change runtime type → T4 GPU`
2. Upload your PDF and reference audio to Google Drive
3. Mount Drive (Cell 3), then use the file browser to copy paths into Cell 4

In [ ]:
# ── Cell 1: Check GPU + Python version ───────────────────────────
import sys, torch

print(f"Python : {sys.version}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU    : {gpu.name}")
    print(f"VRAM   : {gpu.total_memory / 1e9:.1f} GB")
else:
    print("No GPU — go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# ── Cell 2: Install packages ──────────────────────────────────────
%pip install -q chatterbox-tts pdfplumber

# Fix torchvision to match torch 2.6.0 that chatterbox-tts installs.
# cu124 index has torchvision 0.21.0; cu121 caps at 0.20.1.
%pip install -q torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124 --force-reinstall

# !! After this finishes → Runtime → Restart session → re-run this cell → continue from Cell 3

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted. Use the folder icon in the left sidebar to browse files.")

In [ ]:
# ── Cell 4: CONFIG — right-click files in sidebar → Copy path ─────

PDF_PATH   = "/content/drive/MyDrive/audiobook/book.pdf"
VOICE_REF  = "/content/drive/MyDrive/audiobook/reference.mp3"
OUTPUT_DIR = "/content/drive/MyDrive/audiobook/output"

REF_SECONDS = 15    # seconds to trim from reference audio (6-30 ideal)
CHUNK_WORDS = 200   # words per TTS chunk (keep under 250)

In [ ]:
# ── Cell 5: Helper functions ──────────────────────────────────────
import re, subprocess, tempfile, shutil
from pathlib import Path
from collections import Counter
import pdfplumber

CHAPTER_PATTERNS = [
    r"^chapter\s+[\divxlcdm]+", r"^chapter\s+\w+",
    r"^part\s+[\divxlcdm]+",    r"^part\s+\w+",
    r"^\d+\.\s+[A-Z]",
    r"^epilogue$", r"^prologue$", r"^introduction$",
    r"^preface$",  r"^foreword$", r"^appendix", r"^conclusion$",
]

def clean_text(text):
    text = re.sub(r"-\n(\w)", r"\1", text)
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    return text.strip()

def is_chapter_heading(line):
    s = line.strip()
    if not s or len(s) > 80: return False
    return any(re.match(p, s, re.IGNORECASE) for p in CHAPTER_PATTERNS)

def find_repeated_lines(pdf, threshold=5):
    counts = Counter()
    for page in pdf.pages:
        seen = set()
        for line in (page.extract_text() or "").splitlines():
            s = line.strip()
            if s and s not in seen:
                counts[s] += 1; seen.add(s)
    return {l for l, n in counts.items() if n >= threshold}

def extract_chapters(pdf_path):
    chapters, current = [], {"title": "Front Matter", "text": ""}
    with pdfplumber.open(pdf_path) as pdf:
        repeated = find_repeated_lines(pdf)
        for page in pdf.pages:
            for line in (page.extract_text() or "").splitlines():
                s = line.strip()
                if s in repeated: continue
                if is_chapter_heading(s):
                    if current["text"].strip(): chapters.append(current)
                    current = {"title": s, "text": ""}
                else:
                    current["text"] += line + "\n"
    if current["text"].strip(): chapters.append(current)
    return chapters

def split_by_words(text, max_words):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, current, count = [], [], 0
    for s in sentences:
        w = len(s.split())
        if count + w > max_words and current:
            chunks.append(" ".join(current)); current, count = [s], w
        else:
            current.append(s); count += w
    if current: chunks.append(" ".join(current))
    return chunks

def ffmpeg_concat(parts, output):
    with tempfile.TemporaryDirectory() as tmp:
        lst = Path(tmp) / "list.txt"
        lst.write_text("\n".join(f"file '{Path(p).resolve()}'" for p in parts))
        subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0",
                        "-i", str(lst), "-c", "copy", str(output)],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def wav_to_mp3(wav, mp3):
    subprocess.run(["ffmpeg", "-y", "-i", str(wav), "-b:a", "192k", str(mp3)],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    Path(wav).unlink()

print("Helper functions ready.")

In [ ]:
# ── Cell 6: Prepare reference audio ──────────────────────────────
import subprocess
ref_wav = "/content/reference_trimmed.wav"

# Trim to REF_SECONDS starting at 3s (skips any intro noise)
subprocess.run(
    ["ffmpeg", "-y", "-i", VOICE_REF,
     "-ss", "3", "-t", str(REF_SECONDS),
     "-ar", "22050", "-ac", "1", ref_wav],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True
)
print(f"Reference audio trimmed to {REF_SECONDS}s → {ref_wav}")

In [ ]:
# ── Cell 7: Load Chatterbox TTS model ────────────────────────────
# First run downloads the model weights (~1-2 GB)
import torch
from chatterbox.tts import ChatterboxTTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Chatterbox TTS on {device.upper()}...")

model = ChatterboxTTS.from_pretrained(device=device)
print("Model loaded and ready.")

In [ ]:
# ── Cell 8: Extract chapters from PDF ────────────────────────────
print(f"Reading: {PDF_PATH}")
chapters = extract_chapters(PDF_PATH)
print(f"Found {len(chapters)} chapter(s):\n")
for i, ch in enumerate(chapters, 1):
    print(f"  {i:02d}. {ch['title']}  ({len(ch['text'].split()):,} words)")

In [ ]:
# ── Cell 9: Convert chapters to MP3 ──────────────────────────────
import numpy as np
import scipy.io.wavfile as wavfile  # replaces torchaudio — already in Colab

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

for i, ch in enumerate(chapters, 1):
    safe   = re.sub(r"[^\w\s-]", "", ch["title"])[:50].strip()
    fname  = f"{i:02d}_{safe}.mp3"
    out    = out_dir / fname
    text   = clean_text(ch["text"])
    chunks = split_by_words(text, CHUNK_WORDS)

    print(f"[{i}/{len(chapters)}] {ch['title']} — {len(chunks)} chunk(s)")

    with tempfile.TemporaryDirectory() as tmp:
        parts = []
        for j, chunk in enumerate(chunks):
            wav = f"{tmp}/chunk_{j:04d}.wav"
            mp3 = f"{tmp}/chunk_{j:04d}.mp3"
            print(f"  chunk {j+1}/{len(chunks)}...", end="\r")

            # Run inference — clones voice from reference audio
            wav_tensor = model.generate(chunk, audio_prompt_path=ref_wav)

            # Save WAV using scipy (avoids torchaudio version conflicts)
            audio_np = wav_tensor.squeeze().cpu().numpy()
            wavfile.write(wav, model.sr, audio_np)

            wav_to_mp3(wav, mp3)
            parts.append(mp3)

        if len(parts) == 1:
            shutil.copy(parts[0], out)
        else:
            ffmpeg_concat(parts, out)

    size_kb = out.stat().st_size // 1024
    print(f"  → {fname} ({size_kb} KB)          ")

print(f"\nDone! {len(chapters)} MP3(s) saved to {out_dir}")

In [ ]:
# ── Cell 10: (Optional) Stitch all chapters into one file ─────────
full_mp3 = out_dir.parent / f"{Path(PDF_PATH).stem}_full.mp3"
mp3s = sorted(out_dir.glob("*.mp3"))

print(f"Stitching {len(mp3s)} chapter(s)...")
ffmpeg_concat(mp3s, full_mp3)

size_mb = full_mp3.stat().st_size / (1024 * 1024)
print(f"Full audiobook: {full_mp3}  ({size_mb:.1f} MB)")